# HAxBRO KAI — 360M conversation fine-tuning

This notebook fine-tunes the existing `HuggingFaceTB/SmolLM2-360M-Instruct` weights with LoRA. It does not train an 8B model, call a paid inference API, or modify the live HAxBRO app. A free Colab GPU may be available, but Google controls availability and quotas.

**Use Runtime → Change runtime type → T4 GPU** (or another CUDA GPU) before running. The notebook stops if no GPU is available rather than silently attempting a very slow CPU training run.

The included 30-example dataset is a pipeline starter, not enough data to claim broad intelligence gains. Compare the held-out evaluation before and after training.

In [ ]:
import torch, os, pathlib
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import subprocess, os
repo_dir = '/content/haxbro-kai'
if not os.path.isdir(repo_dir):
    subprocess.run(['git','clone','--depth','1','--branch','kai360m-conversation-training','https://github.com/m85948190-ctrl/haxbro.git',repo_dir], check=True)
os.chdir(repo_dir + '/training/kai360m')
print('Working directory:', os.getcwd())


In [ ]:
%pip -q install -r requirements.txt
!python validate_dataset.py


## 1. Record the base-model baseline

This runs the same held-out conversation tests before training. Keep the output for a fair comparison.

In [ ]:
!python evaluate.py --base-only --eval data/eval.jsonl --output outputs/eval-before.jsonl


## 2. Fine-tune the existing 360M model

LoRA updates a small set of trainable adapter weights attached to attention and MLP projections. The base architecture remains 361.8M parameters. The training script calculates loss only on assistant responses, not on user prompts.

In [ ]:
!python train_sft.py --train data/train.jsonl --output-dir outputs/kai360m-conversation-lora --epochs 3 --max-length 1024 --batch-size 2 --gradient-accumulation 8 --learning-rate 2e-4


## 3. Evaluate the trained adapter

Compare `eval-before.jsonl` and `eval-after.jsonl` manually. Do not ship if the model becomes more repetitive, loses instruction-following ability, or performs worse on general questions.

In [ ]:
!python evaluate.py --adapter-dir outputs/kai360m-conversation-lora --eval data/eval.jsonl --output outputs/eval-after.jsonl


## 4. Merge the adapter and package the candidate

The merged model is still the same 360M architecture. This notebook packages the adapter, merged Hugging Face weights, manifest and before/after evaluations. It does not replace the production ONNX model.

In [ ]:
!python merge_adapter.py --adapter-dir outputs/kai360m-conversation-lora --output-dir outputs/kai360m-merged
import zipfile, os
bundle = '/content/kai360m-trained-candidate.zip'
include = ['outputs/kai360m-conversation-lora','outputs/kai360m-merged','outputs/eval-before.jsonl','outputs/eval-after.jsonl']
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in include:
        if os.path.isdir(item):
            for root, _, files in os.walk(item):
                for name in files:
                    path = os.path.join(root, name)
                    z.write(path, os.path.relpath(path, '.'))
        elif os.path.isfile(item):
            z.write(item, item)
print('Candidate bundle:', bundle, 'size MB:', round(os.path.getsize(bundle)/1024/1024, 1))
from google.colab import files
files.download(bundle)


## Release gate

Keep this candidate separate until the before/after tests show a real improvement. Next, export the merged weights to the exact Transformers.js-compatible ONNX/Q4 format used by HAxBRO, verify generation in browser WASM and WebGPU, and test mobile RAM/startup/latency. The 135M fallback stays unchanged. Never describe the model as trained unless the run completed and the output bundle was saved.